# 01 Data sharing agreement — agreement-level boundary

This notebook captures only agreement-level usage metadata. It stays intentionally lightweight so users can approve scope quickly before deeper governance steps in later notebooks.


## 01 Runtime bootstrap and imports

Functions used
- `setup_notebook`: load canonical FabricOps runtime config from `00_env_config`.
- `write_lakehouse_table`: append agreement metadata rows to `METADATA_DATA_AGREEMENT`.
- `register_current_notebook`: register this notebook in `METADATA_NOTEBOOK_REGISTRY`.

You edit
- Agreement-level input values in the next cell.

This notebook produces
- One agreement-level metadata record (optional write).
- One notebook registry record (optional write).


In [ ]:
%run 00_env_config


In [ ]:
from datetime import datetime, timezone

from fabricops_kit import register_current_notebook, setup_notebook, write_lakehouse_table


## 02 Agreement parameters

Provide only agreement-level fields here. Do **not** collect classification/sensitivity/PII at this stage.


In [ ]:
agreement_id = "r002_sales_demo"
agreement_name = "R002 Sales Demo Agreement"
agreement_context = "Demo agreement for FabricOps E2E Test Run 002."
approved_usage = "Use this sample sales order data to test the FabricOps notebook flow."
owner = "FabricOps test owner"
expiry_date = "2026-12-31"
notes = "Agreement-level metadata only. Classification is handled later at table/column level."

# Optional context fields for local testing only (not required for agreement scope).
dataset_name = None
table_name = None

env_name = "dev"
save_to_metadata = True
register_notebook_to_metadata = True


## 03 Build agreement record


In [ ]:
BOOTSTRAP_01 = setup_notebook(
    config=CONFIG,
    env=env_name,
    required_targets=["metadata"],
    notebook_name="01_data_agreement_template",
)
created_at = datetime.now(timezone.utc).isoformat()

agreement_record = {
    "agreement_id": agreement_id,
    "agreement_name": agreement_name,
    "agreement_context": agreement_context,
    "approved_usage": approved_usage,
    "owner": owner,
    "expiry_date": expiry_date,
    "notes": notes,
    "created_at": created_at,
    "environment_name": env_name,
}

if dataset_name:
    agreement_record["dataset_name"] = dataset_name
if table_name:
    agreement_record["table_name"] = table_name

agreement_record


## 04 Save agreement metadata


In [ ]:
if save_to_metadata:
    agreement_df = spark.createDataFrame([agreement_record])
    write_lakehouse_table(
        agreement_df,
        CONFIG,
        env_name,
        "metadata",
        "METADATA_DATA_AGREEMENT",
        mode="append",
    )
    print("Saved agreement metadata to METADATA_DATA_AGREEMENT.")
else:
    print("Dry run: skipped agreement metadata write.")


## 05 Register notebook


In [ ]:
if save_to_metadata and register_notebook_to_metadata:
    register_current_notebook(
        spark,
        metadata_path=CONFIG.path_config.paths[env_name]["metadata"],
        agreement_id=agreement_id,
        notebook_type="01_data_sharing_agreement",
        environment_name=env_name,
    )
    print("Registered notebook in METADATA_NOTEBOOK_REGISTRY.")
else:
    print("Skipped notebook registration.")


## 06 Next step

- `01` creates the agreement-level usage boundary.
- `02` exploration profiles actual tables and columns.
- Table/column classification happens later through governance review.
- `03` pipeline contract enforces approved outputs and rules.
